In [ ]:
"""
	1.	Strip whitespace and lowercase the email column.
	2.	Extract the domain (the part after @) into a new column domain.
	3.	Count how many users per domain.
	4.	Keep only domains with more than 1 user.
	5.	Sort by count descending, then domain name.

Output columns: domain, user_count.
"""

import pandas as pd

users = pd.DataFrame({
    "user_id": [1, 2, 3, 4, 5],
    "email": [
        "  ALICE@example.com  ",
        "bob.smith@work-mail.org",
        "charlie@Example.COM",
        "diana_77@personal.co.uk",
        "eve@example.com"
    ]
})

In [ ]:
"""
From orders (with order_id, customer_id, order_date, status, total_amount):
	1.	Keep only orders that are:
	•	status in ["shipped", "delivered"]
	•	order_date in 2024
	2.	Add a column month as the month name from order_date.
	3.	For each month, calculate:
	•	total revenue
	•	number of unique customers
	4.	Sort months by total revenue descending.
	5.	Output columns: month, total_revenue, unique_customers.
"""

import pandas as pd

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "customer_id": [1, 2, 1, 3, 4, 2, 1, 3],
    "order_date": pd.to_datetime([
        "2024-01-15", "2024-01-22", "2024-02-10", "2024-02-28",
        "2024-03-03", "2024-03-21", "2023-12-30", "2024-03-15"
    ]),
    "status": [
        "shipped", "delivered", "pending", "shipped",
        "delivered", "cancelled", "shipped", "delivered"
    ],
    "total_amount": [200, 150, 300, 400, 250, 500, 180, 350]
})

stati = ['shipped', 'delivered']

filtered = orders[(orders['status'].isin(stati)) & (orders['order_date'].dt.year == 2024)]
filtered = filtered.copy()
filtered['month'] = filtered['order_date'].dt.month_name()

output = (filtered.groupby('month')
          .agg(total_revenue=('total_amount', 'sum'), unique_customers=('customer_id', 'nunique'))
          .sort_values(by='total_revenue', ascending=False)
          .reset_index()
)
output


,month,total_revenue,unique_customers
0,March,600,2
1,February,400,1
2,January,350,2


In [1]:
# Your turn:
	# 2.	Join them so you can see product_name and price_per_unit in the orders data.
	# 3.	Add a total_price column = quantity * price_per_unit.
	# 4.	Group by customer_id and calculate: Total quantity purchased, Total amount spent.
	# 5.	Sort by total amount spent in descending order.

import pandas as pd

# Orders DataFrame
orders_data = {
    "order_id": [1, 2, 3, 4, 5],
    "customer_id": [101, 102, 101, 103, 101],
    "order_date": ["2024-03-01", "2024-03-01", "2024-03-02", "2024-03-02", "2024-03-03"],
    "product_id": ["P1", "P2", "P3", "P1", "P2"],
    "quantity": [1, 2, 1, 1, 3]
}
orders = pd.DataFrame(orders_data)

# Products DataFrame
products_data = {
    "product_id": ["P1", "P2", "P3"],
    "product_name": ["Laptop", "Mouse", "Monitor"],
    "price_per_unit": [900, 20, 200]
}
products = pd.DataFrame(products_data)

merged = pd.merge(orders, products, how='inner', on='product_id')
merged['total_price'] = merged['quantity'] * merged['price_per_unit']
df = (merged.groupby('customer_id').
      agg(total_quantity_purchased=('quantity', 'sum'), 
          total_amount_spent=('total_price', 'sum')).
          reset_index().
          sort_values(by='total_amount_spent', ascending=False))
df 


,customer_id,total_quantity_purchased,total_amount_spent
0,101,5,1160
2,103,1,900
1,102,2,40


In [ ]:
"""
Task: Get the top product per customer by total spending.
	1.	Start from the merged DataFrame (orders + products).
	2.	Calculate total_price (you already know this).
	3.	Group by both customer_id and product_name, summing up total_price.
	4.	For each customer, keep only the product with the highest total spending.
	5.	Sort the final result by customer_id.
"""
merged['total_price'] = merged['quantity'] * merged['price_per_unit']
df = (
    merged.groupby(['customer_id', 'product_name'])['total_price'].
    sum().
    reset_index()
      )
df.groupby('customer_id', group_keys=False).apply(lambda x: x.nlargest(1, 'total_price')).sort_values(by='customer_id').reset_index(drop=True)




/var/folders/wc/8wws1lfd3kv91ykflkc2lzxc0000gn/T/ipykernel_71456/66499867.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('customer_id').apply(lambda x: x.nlargest(1, 'total_price')).reset_index().sort_values(by='customer_id')


ValueError: cannot insert customer_id, already exists

In [ ]:
import pandas as pd

# Step 1: Create the DataFrame
data = {
    "date": ["2024-02-01", "2024-02-01", "2024-02-01",
             "2024-02-02", "2024-02-02", "2024-02-02"],
    "user_id": [1, 2, 3, 1, 2, 3],
    "page_views": [5, 3, 8, 4, 6, 7],
    "clicks": [2, 1, 4, 2, 3, 2]
}

df = pd.DataFrame(data)

# Your turn:
df['date'] = pd.to_datetime(df['date'])
# Step 2: Group by date and calculate totals + CTR: total_page_views, total_clicks
new_df = df.groupby('date').agg(total_page_views=('page_views', 'sum'), total_clicks=('clicks', 'sum')).reset_index()
# Step 3: Sort by CTR (descending): (CTR) = total_clicks / total_page_views.
new_df['ctr'] = new_df['total_clicks'] / new_df['total_page_views']
new_df



,date,total_page_views,total_clicks,ctr
0,2024-02-01,16,7,0.437500
1,2024-02-02,17,7,0.411765


In [89]:
"""
Task:
	1.	Hardcode the data into a DataFrame.
	2.	Calculate CTR per user per day = clicks / page_views.
	3.	Group by date and calculate the average CTR across all users.
	4.	Sort by average CTR in descending order.
"""
new_df = df.groupby(['user_id', 'date']).agg({'clicks': 'sum', 'page_views': 'sum'}).reset_index()
new_df['ctr'] =  new_df['clicks'] / new_df['page_views']
output = new_df.groupby('date').agg(avg_ctr=('ctr', 'mean')).sort_values(by='avg_ctr', ascending=False).reset_index()
output 

,date,avg_ctr
0,2024-02-02,0.428571
1,2024-02-01,0.411111


In [79]:
import pandas as pd

# Step 1: Create the DataFrame
data = {
    "order_id": [1, 2, 3, 4, 5],
    "customer_id": [101, 102, 101, 103, 101],
    "order_date": ["2024-01-02", "2024-01-03", "2024-01-05", "2024-01-06", "2024-01-07"],
    "product": ["Laptop", "Mouse", "Monitor", "Laptop", "Mouse"],
    "quantity": [1, 2, 1, 1, 3],
    "price_per_unit": [900, 20, 200, 950, 18]
}

df = pd.DataFrame(data)

# Step 2: Add total_price column
df['total_price'] = df['quantity'] * df['price_per_unit']
# Step 3: Group by customer_id and calculate totals
new_df = df.groupby('customer_id').agg(total_quantity=('quantity', 'sum'), total_spent=('total_price', 'sum')).sort_values(by='total_spent', ascending=False).reset_index()
# Step 4: Sort by total amount spent (descending)

new_df

,customer_id,total_quantity,total_spent
0,101,5,1154
1,103,1,950
2,102,2,40


In [ ]:
"""
Task:
	1.	For each customer, and for each order date, calculate in the previous 90 days (including that date):
	•	rolling_orders: total number of orders
	•	rolling_spent: total amount spent
	2.	Mark a customer as “rolling loyal” on that date if:
	•	rolling_orders ≥ 3
	•	rolling_spent > the overall average order amount per customer in the dataset
	3.	Return a DataFrame with:
	•	customer_id, order_date, rolling_orders, rolling_spent, is_rolling_loyal
"""
import pandas as pd

sales = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107, 108, 109],
    'customer_id': [1, 1, 2, 3, 2, 4, 3, 1, 2],
    'order_date': pd.to_datetime([
        '2023-01-05', '2023-01-20', '2023-02-02', '2023-02-15', '2023-02-20',
        '2023-03-05', '2023-03-18', '2023-04-02', '2023-04-10'
    ]),
    'amount': [200, 150, 300, 100, 250, 400, 350, 180, 500]
})

sales = sales.sort_values(by=['customer_id', 'order_date'])
df = (sales.groupby(['customer_id']).rolling(window='90D', on='order_date', closed='both').
      agg(
          {'order_id':'count', 'amount':'sum'}).
          rename(columns={'order_id':'rolling_orders', 'amount':'rolling_spent'}).
          reset_index())

df



,order_id,customer_id,order_date,amount
0,101,1,2023-01-05,200
1,102,1,2023-01-20,150
7,108,1,2023-04-02,180
2,103,2,2023-02-02,300
4,105,2,2023-02-20,250
8,109,2,2023-04-10,500
3,104,3,2023-02-15,100
6,107,3,2023-03-18,350
5,106,4,2023-03-05,400


In [21]:
"""
1.	For each customer, calculate:
	•	first_order_date
	•	last_order_date
	•	total_orders (number of orders)
	•	total_spent (sum of amount)
2.	Find loyal customers, defined as customers with at least 2 orders and total_spent above the average total spent across all customers.
3.	Return a final DataFrame with:
	•	customer_id, first_order_date, last_order_date, total_orders, total_spent
	•	is_loyal column (True / False)
"""

import pandas as pd

sales = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107],
    'customer_id': [1, 1, 2, 3, 2, 4, 3],
    'order_date': pd.to_datetime([
        '2023-01-05', '2023-01-20', '2023-02-02',
        '2023-02-15', '2023-02-20', '2023-03-05', '2023-03-18'
    ]),
    'amount': [200, 150, 300, 100, 250, 400, 350]
})

df = (sales.groupby('customer_id').
      agg(first_order_date=('order_date', 'min'), last_order_date=('order_date', 'max'), total_orders=('order_id', 'count'), total_spent=('amount', 'sum')).
      reset_index()
      )

df['is_loyal'] = (df['total_orders'] >= 2) & (df['total_spent'] > df['total_spent'].mean())
df

,customer_id,first_order_date,last_order_date,total_orders,total_spent,is_loyal
0,1,2023-01-05,2023-01-20,2,350,False
1,2,2023-02-02,2023-02-20,2,550,True
2,3,2023-02-15,2023-03-18,2,450,True
3,4,2023-03-05,2023-03-05,1,400,False


In [7]:
"""
	1.	For each month, compute:
	•	monthly_orders: total number of orders
	•	unique_customers: number of distinct customers
	•	total_revenue: total amount spent
	2.	Identify “peak months” where:
	•	Revenue is above the average monthly revenue
	3.	Return a final DataFrame with:
	•	month (as YYYY-MM)
	•	monthly_orders, unique_customers, total_revenue
	•	is_peak: True if it’s a peak month, else False
"""
import pandas as pd
import numpy as np

orders = pd.DataFrame({
    'order_id': [601, 602, 603, 604, 605, 606, 607, 608, 609, 610],
    'customer_id': [1, 2, 1, 3, 2, 3, 1, 2, 3, 1],
    'order_date': pd.to_datetime([
        '2023-01-10', '2023-01-20', '2023-02-15', '2023-02-20', '2023-03-05',
        '2023-03-10', '2023-04-01', '2023-04-05', '2023-04-15', '2023-05-01'
    ]),
    'amount': [80, 100, 90, 120, 110, 130, 150, 140, 160, 170]
})

monthlies = (orders.groupby(orders['order_date'].dt.to_period('M').astype(str)).
             agg(monthly_orders=('order_id', 'count'), unique_customers=('customer_id', 'nunique'), total_revenue=('amount', 'sum')).         
             reset_index().
             rename(columns={'order_date': 'month'})
             )

monthlies['is_peak'] = monthlies['total_revenue'] > monthlies['total_revenue'].mean()
monthlies

,month,monthly_orders,unique_customers,total_revenue,is_peak
0,2023-01,2,2,180,False
1,2023-02,2,2,210,False
2,2023-03,2,2,240,False
3,2023-04,3,3,450,True
4,2023-05,1,1,170,False


In [ ]:
"""
🧠 Task:
	1.	For each customer, compute:
	•	total_orders: total number of orders
	•	total_spent: total amount spent
	•	first_order: date of their first order
	•	last_order: date of their last order
	2.	Create a column customer_status using multi-condition logic:
	•	'new' if first_order is in April 2023 or later
	•	'churned' if last_order is before March 1, 2023
	•	'active' otherwise

"""
import pandas as pd
import numpy as np

orders = pd.DataFrame({
    'order_id': [601, 602, 603, 604, 605, 606, 607, 608, 609, 610],
    'customer_id': [1, 2, 1, 3, 2, 3, 1, 2, 3, 1],
    'order_date': pd.to_datetime([
        '2023-01-10', '2023-01-20', '2023-02-15', '2023-02-20', '2023-03-05',
        '2023-03-10', '2023-04-01', '2023-04-05', '2023-04-15', '2023-05-01'
    ]),
    'amount': [80, 100, 90, 120, 110, 130, 150, 140, 160, 170]
})

# grouped = orders.groupby('customer_id').agg(
#     total_orders=('order_id', 'count'),
#     total_spent=('amount', 'sum'),
#     first_order=('order_date', 'min'),
#     last_order=('order_date', 'max')
# ).reset_index()

# conditions = [
#     grouped['first_order'] >= pd.to_datetime('2023-04-01'),
#     grouped['last_order'] < pd.to_datetime('2023-03-01')
# ]
# choices = ['new', 'churned']

# grouped['customer_status'] = np.select(conditions, choices, default='active')
# grouped





,customer_id,total_orders,total_spent,first_order,last_order,customer_status
0,1,4,490,2023-01-10,2023-05-01,active
1,2,3,350,2023-01-20,2023-04-05,active
2,3,3,410,2023-02-20,2023-04-15,active


In [1]:
"""
🧠 Task:
	1.	Filter: Keep only customers who signed up before March 1, 2023.
	2.	Join: Merge with orders to get their order history.
	3.	Compute per customer:
	•	order_count: total number of orders
	•	total_spent: total amount spent
	4.	Add: A column loyalty_tag:
	•	"high" if total_spent ≥ 400
	•	"medium" if 200 ≤ total_spent < 400
	•	"low" otherwise
"""

import pandas as pd

customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'signup_date': pd.to_datetime(['2022-12-01', '2023-01-15', '2023-02-10', '2023-04-01'])
})

orders = pd.DataFrame({
    'order_id': [501, 502, 503, 504, 505, 506, 507, 508],
    'customer_id': [1, 2, 1, 3, 2, 3, 1, 4],
    'order_date': pd.to_datetime([
        '2023-03-01', '2023-03-05', '2023-03-20', '2023-03-25',
        '2023-04-05', '2023-04-10', '2023-04-15', '2023-04-20'
    ]),
    'amount': [100, 200, 150, 120, 180, 130, 160, 90]
})

filtered = customers[customers['signup_date'] < pd.Timestamp('2023-03-01')]
filtered = filtered.copy()
merged = pd.merge(filtered, orders, how='inner', on='customer_id')
grouped = merged.groupby('customer_id').agg(order_count=('order_id', 'count'), total_spent=('amount', 'sum')).reset_index()
grouped['loyalty_tag'] = ['high' if x >= 400 else 'medium' if x >= 200 else 'low' for x in grouped['total_spent']]
grouped


,customer_id,order_count,total_spent,loyalty_tag
0,1,3,410,high
1,2,2,380,medium
2,3,2,250,medium


In [309]:
"""
	1.	Filter orders from March 2023 onward
	2.	For each customer, calculate:
	•	recent_order_count: number of orders since March 1, 2023
	•	recent_total_spent: total amount spent since that date
	3.	Add a boolean column is_recent_big_spender:
	•	True if recent_total_spent > $250
	•	False otherwise
"""


import pandas as pd

orders = pd.DataFrame({
    'order_id': [401, 402, 403, 404, 405, 406, 407, 408, 409, 410],
    'customer_id': [1, 2, 1, 3, 2, 3, 1, 2, 3, 1],
    'order_date': pd.to_datetime([
        '2023-02-15', '2023-02-20', '2023-03-05', '2023-03-10', '2023-03-15',
        '2023-03-20', '2023-04-01', '2023-04-10', '2023-04-15', '2023-04-20'
    ]),
    'amount': [80, 120, 150, 90, 160, 110, 200, 100, 130, 180]
})

filtered = orders[orders['order_date'] >= '2023-03-01']
filtered = filtered.copy()
df = filtered.groupby('customer_id').agg(recent_order_count=('order_id', 'count'), recent_total_spent=('amount', 'sum')).reset_index()
# df['is_recent_big_spender'] = df['recent_total_spent'].apply(lambda x: x > 250)
df['is_recent_big_spender'] = df['recent_total_spent'] > 250

df 

,customer_id,recent_order_count,recent_total_spent,is_recent_big_spender
0,1,3,530,True
1,2,2,260,True
2,3,3,330,True


In [294]:
"""
🧠 Problem: High-Value Repeat Customers
🎯 Tasks:
	1.	Create a customer-level summary with:
	•	total number of orders
	•	total amount spent
	2.	Add a boolean flag called is_high_value_repeat:
	•	True if a customer has more than 2 orders and total amount spent is over $300
	•	False otherwise
"""
import pandas as pd

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'customer_id': [1, 1, 2, 2, 3, 1, 2, 3],
    'order_date': pd.to_datetime([
        '2023-01-10', '2023-01-17', '2023-01-12',
        '2023-01-20', '2023-01-25', '2023-02-01',
        '2023-02-05', '2023-02-10'
    ]),
    'amount': [120, 80, 150, 200, 50, 70, 180, 30]
})

summary = orders.groupby('customer_id').agg({'order_id': 'count', 'amount': 'sum'}).reset_index().rename(columns={'order_id': 'order_count', 'amount': 'total_spent'})
summary['is_high_value_repeat'] = ((summary['order_count'] > 2)  & (summary['total_spent'] > 300))
summary

,customer_id,order_count,total_spent,is_high_value_repeat
0,1,3,270,False
1,2,3,530,True
2,3,2,80,False


In [ ]:
"""
Tasks:
	1.	Total amount spent per customer
	2.	Number of orders per customer
	3.	Customer with the highest average order value
	4.	For each customer, days between their first and last order
"""
import pandas as pd

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106],
    'customer_id': [1, 1, 2, 2, 3, 1],
    'order_date': pd.to_datetime([
        '2023-01-10', '2023-01-17', '2023-01-12',
        '2023-01-20', '2023-01-25', '2023-02-01'
    ]),
    'amount': [120, 80, 150, 200, 50, 70]
})

specs = orders.groupby('customer_id').agg({'amount': 'sum', 'order_id': 'count'}).reset_index().rename(columns={'amount': 'total_amount', 'order_id': 'order_count'})

highest_avg = orders.groupby('customer_id')['amount'].mean().reset_index().nlargest(1, 'amount')
# specs['highest'] = orders['customer_id'] == highest_avg['customer_id'].max()

days_between = orders.groupby('customer_id')['order_date'].agg(['min','max']).reset_index().rename(columns={'min': 'first_order_date', 'max': 'last_order_date'})
days_between['days_between'] = (days_between['last_order_date'] - days_between['first_order_date']).dt.days

merged = pd.merge(specs, days_between, how='inner', on='customer_id')
merged




,customer_id,total_amount,order_count,highest,first_order_date,last_order_date,days_between
0,1,270,3,False,2023-01-10,2023-02-01,22
1,2,350,2,False,2023-01-12,2023-01-20,8
2,3,50,1,True,2023-01-25,2023-01-25,0


In [ ]:
"""
Task:
Filter df to keep only rows where the 'text' contains a 4-digit number starting with '20' (like a year).
Pattern: '20\d{2}'
Give it a try using .str.contains() with regex=True.
"""



In [210]:
import pandas as pd
import re 

df = pd.DataFrame({
    'text': ['Order 123 shipped on 2023-07-15', 'Invoice #4567 dated 2022/12/01', 'Ref: 89, due 01-01-2025']
})

# Create a new column 'numbers' that contains a list of all number groups (i.e., sequences of digits) found in each row.

df['numbers'] = df['text'].apply(lambda x: re.findall('[0-9]+', x))

df['output'] = df['numbers'].apply(lambda x: x if len(x) > 3 else 0)
df 


# for t in text 
# extract numbers with findall() (set to int?)
# if len(num) = 4 and startswith? 20
# include 

# df[len(df['text']) > 20]

,text,numbers,output
0,Order 123 shipped on 2023-07-15,"[123, 2023, 07, 15]","[123, 2023, 07, 15]"
1,Invoice #4567 dated 2022/12/01,"[4567, 2022, 12, 01]","[4567, 2022, 12, 01]"
2,"Ref: 89, due 01-01-2025","[89, 01, 01, 2025]","[89, 01, 01, 2025]"


In [193]:
import pandas as pd 
import re 

df = pd.DataFrame({
    'code': [' abc-123 ', 'X#YZ_789', '  42*gold!', ' cleanCode99 ']
})

# df['clean_code'] = df['code'].apply(lambda x: re.sub('[^a-zA-Z0-9]', '', x.strip().lower()))

# From the same df['code'], extract only the numeric part into a new column 'digits'.

df['digits'] = df['code'].apply(lambda x: re.sub('[^0-9]', '', x))
df

,code,digits
0,abc-123,123
1,X#YZ_789,789
2,42*gold!,42
3,cleanCode99,99


In [188]:
"""
Question:
Clean the 'raw' column by:
	1.	Stripping leading/trailing whitespace
	2.	Removing all non-alphabetic characters
	3.	Converting to lowercase
"""
import pandas as pd
import re 

df = pd.DataFrame({
    'raw': ['  hello ', 'WORLD!!', '  PyThOn3  ', '@data_ ', ' clean ']
})

# df['cleaned'] = df['raw'].str.replace('[^A-Za-z0-9]', '', regex=True).str.lower()

df['cleaned'] = df['raw'].apply(lambda x: re.sub('[^A-Za-z0-9]', '', x.strip().lower()))




df 

,raw,cleaned
0,hello,hello
1,WORLD!!,world
2,PyThOn3,python3
3,@data_,data
4,clean,clean


In [72]:
"""
Task: Pivot this data so each student’s name is a row, each subject is a column, and the cell values are the scores.

Expected format:
subject   English  Math
name                    
Alice          92    85
Bob            67    78
Charlie        88    90
"""
import pandas as pd 

data = {
    'name': ['Alice', 'Bob', 'Alice', 'Bob', 'Charlie', 'Charlie'],
    'subject': ['Math', 'Math', 'English', 'English', 'Math', 'English'],
    'score': [85, 78, 92, 67, 90, 88]
}
df = pd.DataFrame(data)

# pivoted = df.pivot(index='name', columns='subject', values='score').reset_index()
# pivoted
# pivoted.groupby('name', as_index=True)['score'].mean()
df.groupby('name')['score'].mean().reset_index()

,name,score
0,Alice,88.5
1,Bob,72.5
2,Charlie,89.0


In [56]:
"""
Question:
Use pandas to filter this DataFrame to only include rows where the score is ≥ 70.
"""
import pandas as pd

names = ['Alice', 'Bob', 'Charlie', 'Dana']
scores = [85, 62, 90, 58]
df = pd.DataFrame({'name': names, 'score': scores})

# df[df['score'] >= 70]

"""
Task: Add a new column to the DataFrame called 'result' that contains 'Pass' if the score is ≥ 70, else 'Fail'.
"""

# df['result'] = ['Pass' if s >= 70 else 'Fail' for s in df['score']]

"""
Recreate the 'result' column using .apply() with a lambda function instead of list comprehension.
"""

df['result'] = df['score'].apply(lambda x: 'Pass' if x >= 70 else 'Fail')
df



,name,score,result
0,Alice,85,Pass
1,Bob,62,Fail
2,Charlie,90,Pass
3,Dana,58,Fail


In [ ]:
"""
🧠 Problem: Purchase History by Customer
🛠️ Task:
Use list comprehension to create a list of dictionaries like:

[
  {'customer': 'Alice', 'products': ['Widget', 'Gizmo']},
  {'customer': 'Bob', 'products': ['Gadget', 'Widget']},
  {'customer': 'Dan', 'products': ['Gizmo']},
  {'customer': 'Eve', 'products': ['Gadget', 'Widget']}
]
"""
import pandas as pd

df = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Alice', 'Dan', 'Bob', 'Eve', 'Eve'],
    'product': ['Widget', 'Gadget', 'Gizmo', 'Gizmo', 'Widget', 'Gadget', 'Widget']
})

dict = {}
list_dict = []

for index, rows in df.iterrows():
    list_dict.get('index') = 
    list_dict.append(rows.tolist())

# list_dict 

df.values.tolist()


output:
{'data': 3, 'is': 3, 'powerful': 1, 'python': 1, 'great': 1, 'for': 1, 'analysis': 2, 'fun': 1}
"""
sentences = [
    "data is powerful",
    "python is great for data analysis",
    "data analysis is fun"
]

output = {}
for s in sentences:
    spl = s.split()
    for w in spl:
        output[w] = output.get(w, 0) + 1
        
# output

[['Alice', 'Widget'],
 ['Bob', 'Gadget'],
 ['Alice', 'Gizmo'],
 ['Dan', 'Gizmo'],
 ['Bob', 'Widget'],
 ['Eve', 'Gadget'],
 ['Eve', 'Widget']]

In [52]:
"""
💼 Problem: Customer Risk Tagging
🛠️ Task:
	1.	Compute total amount and total chargebacks per customer.
	2.	Add a column 'risk' to df using list comprehension, with the following rule:
	•	'high' if total chargebacks ≥ 2 and total amount > 100
	•	'low' otherwise
"""
import pandas as pd

df = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Alice', 'Dan', 'Bob', 'Eve', 'Eve'],
    'amount': [120, 50, 80, 40, 70, 200, 150],
    'chargebacks': [0, 1, 0, 0, 2, 1, 3]
})

df['tot_amount'] = df.groupby('customer')['amount'].transform('sum')
df['tot_chargebacks'] = df.groupby('customer')['chargebacks'].transform('sum')
df['risk'] = ['high' if c >= 2 and a > 100 else 'low' for a,c in zip(df['tot_amount'], df['tot_chargebacks'])]
df 



,customer,amount,chargebacks,tot_amount,tot_chargebacks,risk
0,Alice,120,0,200,0,low
1,Bob,50,1,120,3,high
2,Alice,80,0,200,0,low
3,Dan,40,0,40,0,low
4,Bob,70,2,120,3,high
5,Eve,200,1,350,4,high
6,Eve,150,3,350,4,high


In [49]:
"""
📦 Problem: Tagging High-Spending Customers
🛠️ Task:
	1.	Compute total amount spent per customer.
	2.	Using list comprehension, create a new column 'tag' in the original df:
	•	'high' if the customer’s total spend is ≥ 200
	•	'low' otherwise
"""
import pandas as pd

df = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Alice', 'Dan', 'Bob', 'Eve', 'Eve'],
    'amount': [120, 50, 80, 40, 70, 200, 150]
})

# df['spend'] = df.groupby('customer')['amount'].transform('sum')

df['tag'] = ['high' if x >= 200 else 'low' for x in df.groupby('customer')['amount'].transform('sum')]
df
# data = list(zip(df['customer'], df['amount']))
# data


,customer,amount,tag
0,Alice,120,high
1,Bob,50,low
2,Alice,80,high
3,Dan,40,low
4,Bob,70,low
5,Eve,200,high
6,Eve,150,high


In [41]:
"""
🧪 Problem: Scholarship Eligibility
🛠️ Task:
Using list comprehension, create a new column 'eligible' with these rules:
	•	If (score + extra_credit) >= 90 and discipline_flag == 0, then 'yes'
	•	Otherwise, 'no'
"""
import pandas as pd

df = pd.DataFrame({
    'student': ['Amy', 'Ben', 'Cara', 'Dan', 'Eve'],
    'score': [88, 92, 70, 65, 85],
    'extra_credit': [5, 0, 10, 15, 0],
    'discipline_flag': [0, 1, 0, 0, 1]  # 0 = good, 1 = flagged
})

df['eligible'] = ['yes' if s + e >= 90 and d == 0
                  else 'no'
                  for s, e, d in zip(df['score'], df['extra_credit'], df['discipline_flag'])]
df 

,student,score,extra_credit,discipline_flag,eligible
0,Amy,88,5,0,yes
1,Ben,92,0,1,no
2,Cara,70,10,0,no
3,Dan,65,15,0,no
4,Eve,85,0,1,no


In [40]:
"""
🧠 Problem: Pass/Fail with Bonus Rule
🛠️ Task:
Using list comprehension, add a new column 'result' based on the following rule:
	•	If score + extra_credit ≥ 60, label as "pass"
	•	Otherwise, label as "fail"
"""
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'student': ['Alex', 'Bri', 'Chris', 'Dana', 'Eli'],
    'score': [55, 62, 49, 85, 58],
    'extra_credit': [10, 0, 5, 0, 7]
})

# df['result'] = np.where(df['score'] + df['extra_credit'] >= 60, 'pass', 'fail')
df['result'] = ['pass' if s + e >= 60
                else 'fail'
                for s,e in zip(df['score'], df['extra_credit'])
                ]
df


,student,score,extra_credit,result
0,Alex,55,10,pass
1,Bri,62,0,pass
2,Chris,49,5,fail
3,Dana,85,0,pass
4,Eli,58,7,pass


In [36]:
"""
📦 Problem: Tagging Products Based on Sales
🛠️ Task:
Using list comprehension, add a new column 'tag' with these rules:
	•	"bestseller" if units_sold ≥ 100
	•	"average" if 50 ≤ units_sold < 100
	•	"low" if less than 50
"""
import pandas as pd

df = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Gizmo', 'Thingy', 'Doodad'],
    'units_sold': [120, 35, 200, 15, 80]
})

df['tag'] = ['bestseller' if x >= 100 
             else 'average' if x < 100 and x >= 50 
             else 'low' 
             for x in df['units_sold']]
df 


,product,units_sold,tag
0,Widget,120,bestseller
1,Gadget,35,low
2,Gizmo,200,bestseller
3,Thingy,15,low
4,Doodad,80,average


In [ ]:
"""
📊 Problem: Customer Product Summary
🛠️ Task:
	1.	Create a list of unique customers.
	2.	Use list comprehension to build a list of tuples, where each tuple is:

(customer_name, total_units_ordered)

	3.	Convert that list into a new DataFrame with columns: 'customer', 'total_quantity'.
"""
import pandas as pd

df = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Alice', 'David', 'Bob', 'Alice'],
    'product': ['Widget', 'Gadget', 'Gizmo', 'Gizmo', 'Widget', 'Gadget'],
    'quantity': [3, 2, 1, 4, 2, 5]
})

new_df = df.groupby('customer')['quantity'].sum().reset_index()

tuple_list = [tuple(row) for index, row in new_df.iterrows()]
column_names = ['customer', 'total_quantity']

output = pd.DataFrame(tuple_list, columns=column_names)
output





,customer,total_quantity
0,Alice,9
1,Bob,4
2,David,4


In [ ]:
"""
🧪 Problem: Analyze Sales Data
Task:
	1.	Add a column total which is quantity * price.
	2.	Convert date to a datetime object.
	3.	Find the total sales per customer.
	4.	Find the top 3 products by total sales.
	5.	Filter to include only orders in 2023.
"""
import pandas as pd

data = {
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'customer': ['Alice', 'Bob', 'Alice', 'David', 'Bob', 'Alice'],
    'product': ['Widget', 'Gadget', 'Widget', 'Gizmo', 'Gadget', 'Gizmo'],
    'quantity': [3, 2, 5, 1, 4, 2],
    'price': [20.0, 15.0, 20.0, 50.0, 15.0, 50.0],
    'date': ['2023-01-15', '2023-02-10', '2022-12-20', '2023-03-05', '2023-04-12', '2022-11-01']
}

df = pd.DataFrame(data)

# 1. 
df['total'] = df['quantity'] * df['price']

# 2. 
df['date'] = pd.to_datetime(df['date'])

# 3. 
sales_per_cust = df.groupby('customer')['total'].sum()

# 4. 
top_3 = df.groupby('product')['total'].sum().nlargest(3)

# 5. 
orders_2023 = df[df['date'].dt.year == 2023]


,order_id,customer,product,quantity,price,date,total
0,1001,Alice,Widget,3,20.0,2023-01-15,60.0
1,1002,Bob,Gadget,2,15.0,2023-02-10,30.0
3,1004,David,Gizmo,1,50.0,2023-03-05,50.0
4,1005,Bob,Gadget,4,15.0,2023-04-12,60.0


In [ ]:
"""
🎯 Task: Add a new column 'domain' using a list comprehension that extracts just the domain name (e.g., 'gmail.com', 'yahoo.com', etc.).
"""
import pandas as pd

df = pd.DataFrame({
    'email': ['alice@gmail.com', 'bob@yahoo.com', 'cathy@outlook.com', 'david@gmail.com']
})
# extract...split string, get index == 1

df['domain'] = [x.split('@')[-1] for x in df['email']]

,email,domain
0,alice@gmail.com,gmail.com
1,bob@yahoo.com,yahoo.com
2,cathy@outlook.com,outlook.com
3,david@gmail.com,gmail.com


In [50]:
"""
🔤 List Comprehension Practice: Name Flags
Create a new column 'proper_case' using a list comprehension:
	•	True if the name is in title case (first letter capitalized in each word)
	•	False otherwise

"""
import pandas as pd

df = pd.DataFrame({
    'name': ['Alice Smith', 'bob jones', 'Cathy Lee', 'david brown', 'Eva Green']
})

df['proper_case'] = [x.istitle() for x in df['name']]
df


,name,proper_case
0,Alice Smith,True
1,bob jones,False
2,Cathy Lee,True
3,david brown,False
4,Eva Green,True


In [ ]:
"""
🧠 List Comprehension Practice: Product Tags
Add a new column 'tag' using a list comprehension with this logic:
	•	"Expensive" if price > 500
	•	"Mid" if price is between 100 and 500 (inclusive)
	•	"Cheap" if price < 100
"""
import pandas as pd

df = pd.DataFrame({
    'product': ['Desk', 'Chair', 'iPad', 'Monitor', 'Notebook', 'Mouse'],
    'price': [200, 100, 799, 250, 5, 40]
})

df['tag'] = ['Expensive' if x > 500 else 'Mid' if x >= 100 and x < 500 else 'Cheap' for x in df['price']]
df

,product,price,tag
0,Desk,200,Mid
1,Chair,100,Mid
2,iPad,799,Expensive
3,Monitor,250,Mid
4,Notebook,5,Cheap
5,Mouse,40,Cheap


In [41]:
"""
📊 Practice Problem: Top Store per Region
🎯 Tasks:
	1.	Find the store with the highest sales in each region.
	2.	Return a new DataFrame with just those top-performing stores.
"""
import pandas as pd

df = pd.DataFrame({
    'store': ['A', 'B', 'C', 'D', 'E', 'F'],
    'region': ['North', 'North', 'South', 'South', 'East', 'East'],
    'sales': [5000, 7000, 6000, 3000, 4000, 9000]
})


# df_largest = df.nlargest(1, 'sales')
# df['max_sales'] = 
# df.groupby('region')['sales'].nlargest(1)

df.groupby('region', group_keys=False).apply(lambda g: g.nlargest(1, 'sales'))

/var/folders/wc/8wws1lfd3kv91ykflkc2lzxc0000gn/T/ipykernel_12828/1797184555.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('region', group_keys=False).apply(lambda g: g.nlargest(1, 'sales'))


,store,region,sales
5,F,East,9000
1,B,North,7000
2,C,South,6000


In [23]:
"""
📊 Practice Problem: Product vs. Category Performance

🎯 Tasks:
	1.	Compute the average sales per category
	2.	Join that info back to the original DataFrame
	3.	Add a new column 'above_avg' that is True if the product’s sales are above the category average
"""
import pandas as pd 

df = pd.DataFrame({
    'product_id': ['p1', 'p2', 'p3', 'p4', 'p5', 'p6'],
    'category': ['Books', 'Books', 'Tech', 'Tech', 'Tech', 'Tech'],
    'sales': [120, 150, 1000, 900, 300, 500]
})

df['avg_sales']= df.groupby('category')['sales'].transform('mean')
df['above_avg'] = df['sales'] > df['avg_sales']
df 


# # merge and create new df
# merged = pd.merge(df, new_df, how='inner', on='category')
# merged
# # create new column

,product_id,category,sales,avg_sales,above_avg
0,p1,Books,120,135.0,False
1,p2,Books,150,135.0,True
2,p3,Tech,1000,675.0,True
3,p4,Tech,900,675.0,True
4,p5,Tech,300,675.0,False
5,p6,Tech,500,675.0,False


In [8]:
"""
📊 Practice Problem: Seller Discount Strategy

🎯 Tasks:
	1.	For each seller_id, calculate the average discount_rate and total sales.
	2.	Add a column high_discount_seller that is True if their average discount is >= 0.10

Result:

seller_id | avg_discount | total_sales | high_discount_seller

"""
import pandas as pd

df = pd.DataFrame({
    'seller_id': ['s1', 's1', 's2', 's2', 's3', 's3', 's3'],
    'product_id': ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7'],
    'discount_rate': [0.10, 0.0, 0.15, 0.05, 0.0, 0.2, 0.25],
    'sales': [100, 150, 80, 120, 200, 90, 60]
})

new_df = df.groupby('seller_id').agg({'discount_rate': 'mean', 'sales': 'sum'}).rename(columns={'discount_rate': 'avg_discount', 'sales': 'total_sales'}).reset_index()
new_df['high_discount_seller'] = new_df['avg_discount'].apply(lambda x: x >= 0.10)
new_df


# new_df['at_risk'] = new_df['avg_discount'].apply(lambda x: x < 70)




,seller_id,avg_discount,total_sales,high_discount_seller
0,s1,0.05,250,False
1,s2,0.10,200,True
2,s3,0.15,350,True


In [15]:
"""
📊 Practice Problem: Percentile Rank Within Category
🎯 Task:
	1.	For each category, compute the percentile rank of each product based on sales, scaled from 0 to 1, where:
	•	Highest sales = 1.0
	•	Lowest sales = 0.0
	2.	Add a new column: 'percentile_rank'
"""
import pandas as pd 

df = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5, 6, 7],
    'category': ['Books', 'Books', 'Books', 'Tech', 'Tech', 'Tech', 'Tech'],
    'sales': [120, 340, 200, 1500, 800, 600, 400]
})

df['cat_ntile'] = df.groupby('category')['sales'].rank(pct=True, ascending=True)
df

,product_id,category,sales,cat_ntile
0,1,Books,120,0.333333
1,2,Books,340,1.000000
2,3,Books,200,0.666667
3,4,Tech,1500,1.000000
4,5,Tech,800,0.750000
5,6,Tech,600,0.500000
6,7,Tech,400,0.250000


In [12]:
"""
📊 Practice Problem: Rank Products by Popularity Within Category
🎯 Tasks:
	1.	For each category, rank the products by sales (highest = rank 1).
	2.	Add a new column called 'popularity_rank'.
	3.	Return a DataFrame with:

product_id | category | sales | popularity_rank
"""
import pandas as pd 

df = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5, 6, 7],
    'category': ['Books', 'Books', 'Books', 'Tech', 'Tech', 'Tech', 'Tech'],
    'sales': [120, 340, 200, 1500, 800, 600, 400]
})

df['popularity_rank'] = df.groupby('category')['sales'].rank(method='dense', ascending=False)
df

,product_id,category,sales,popularity_rank
0,1,Books,120,3.0
1,2,Books,340,1.0
2,3,Books,200,2.0
3,4,Tech,1500,1.0
4,5,Tech,800,2.0
5,6,Tech,600,3.0
6,7,Tech,400,4.0


In [8]:
"""
📊 Practice Problem: Product Rating Cleaner
🎯 Tasks:
	1.	Drop products with no reviews (i.e., reviews == 0)
	2.	Fill missing rating values with the average rating per product
	3.	Return a DataFrame with:
	•	product
	•	rating_filled
	•	reviews
"""
import pandas as pd

df = pd.DataFrame({
    'product': ['A', 'A', 'A', 'B', 'B', 'C', 'C', 'D', 'E'],
    'rating': [4.5, 4.7, None, 3.0, None, 4.0, 4.1, None, None],
    'reviews': [10, 8, 5, 2, 1, 20, 15, 0, 0]
})

new_df = df[df['reviews'] > 0].copy()
new_df['rating'] = new_df['rating'].fillna(new_df.groupby('product')['rating'].transform('mean'))
new_df 

,product,rating,reviews
0,A,4.5,10
1,A,4.7,8
2,A,4.6,5
3,B,3.0,2
4,B,3.0,1
5,C,4.0,20
6,C,4.1,15


In [177]:
"""
📊 Practice Problem: Incomplete Survey Cleanup

🎯 Tasks:
	1.	Drop rows where all values except 'respondent' are missing.
	2.	Fill missing age values with the mean age.
	3.	Fill missing income with the median income.
	4.	Fill missing country with the string "Unknown"
"""
import pandas as pd 

df = pd.DataFrame({
    'respondent': [1, 2, 3, 4, 5],
    'age': [25, None, 30, None, 22],
    'income': [50000, 60000, None, None, 45000],
    'country': ['US', 'Canada', 'US', None, 'Canada']
})

new_df = df[((df['age'].notna()) | (df['income'].notna()) | (df['country'].notna())) 
            & (df['respondent'].notna())]

new_df = new_df.copy()

new_df['age'] = new_df['age'].fillna(new_df['age'].mean())
new_df['income'] = new_df['income'].fillna(new_df['income'].median())
new_df['country'] = new_df['country'].fillna('Unknown')

new_df



,respondent,age,income,country
0,1,25.000000,50000.0,US
1,2,25.666667,60000.0,Canada
2,3,30.000000,50000.0,US
4,5,22.000000,45000.0,Canada


In [163]:
import pandas as pd

df = pd.DataFrame({
    'employee': ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob', 'Charlie'],
    'department': ['HR', 'HR', 'Finance', 'HR', 'HR', 'Finance'],
    'year': [2022, 2022, 2022, 2023, 2023, 2023],
    'score': [85, 88, 90, 87, 84, 91]
})

# new_df = df.groupby(['department', 'year'])['score'].mean().reset_index()

pd.pivot_table(df, index=['department'], columns='year', values='score', aggfunc='mean').reset_index()

year,department,2022,2023
0,Finance,90.0,91.0
1,HR,86.5,85.5


In [145]:
"""
📊 Practice Problem: Course Completion Tracker

🎯 Tasks:
	1.	Filter to only rows where "completed" == True
	2.	For each student, calculate:
	•	The number of completed courses
	•	Their average score across completed courses
	3.	Return a DataFrame with:

student | courses_completed | avg_score
"""

import pandas as pd

df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob', 'Charlie', 'Alice'],
    'course': ['Math', 'Math', 'Math', 'English', 'English', 'English', 'History'],
    'completed': [True, True, False, True, False, True, False],
    'score': [88, 76, None, 92, None, 85, None]
})

(
    df[df['completed']]
    .groupby('student')
    .agg({'course': 'count', 'score': 'mean'})
    .reset_index()
    .rename(columns={'course': 'course_count', 'score': 'avg_score'}))

,student,course_count,avg_score
0,Alice,2,90.0
1,Bob,1,76.0
2,Charlie,1,85.0


In [139]:
"""
🎯 Tasks:
	1.	Merge the two DataFrames to include employee name, department, year, and score.
	2.	Compute the average review score per department.
	3.	Output a DataFrame:

department | avg_score

"""
import pandas as pd

employees = pd.DataFrame({
    'emp_id': [101, 102, 103, 104, 105],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eli'],
    'department': ['HR', 'IT', 'IT', 'Finance', 'HR']
})

reviews = pd.DataFrame({
    'emp_id': [101, 102, 102, 104, 105, 105],
    'year': [2022, 2022, 2023, 2022, 2022, 2023],
    'score': [3.9, 4.2, 4.5, 3.5, 4.0, 4.1]
})


df = pd.merge(employees, reviews, how="inner", on='emp_id', suffixes=('_x','_y'))
df.groupby('department')['score'].mean().reset_index().rename(columns={'score': 'avg_score'})


,department,avg_score
0,Finance,3.50
1,HR,4.00
2,IT,4.35


In [ ]:
"""
📊 Practice Problem: Review Scores Summary

🎯 Tasks:
	1.	Filter the DataFrame to include only verified reviews.
	2.	Group by "product" and calculate the average review score.
	3.	Return a DataFrame sorted by average score descending, with columns:

product | avg_score
"""

import pandas as pd

df = pd.DataFrame({
    'product': ['A', 'A', 'A', 'B', 'B', 'C', 'C', 'C', 'C'],
    'review_score': [5, 4, 5, 3, 2, 4, 4, 5, 3],
    'verified': [True, False, True, True, False, True, False, True, True]
})

new_df = df[df['verified'] == True].groupby('product')['review_score'].mean().rename(columns={'review_score': 'avg_score'}).sort_values(by='avg_score', ascending=False).reset_index()
new_df[['product', 'avg_score']]

,product,avg_score
0,A,5.0
1,C,4.0
2,B,3.0


In [124]:
"""
🧠 Practice: Normalize and Filter Words

🎯 Task:
	1.	Normalize each sentence:
	•	Remove extra spaces
	•	Convert all words to lowercase
	2.	Flatten all the words into one list
	3.	Keep only words longer than 3 characters

⸻

Expected output (approx):

['data', 'powerful', 'python', 'great', 'data', 'analysis', 'data', 'analysis']

"""

sentences = [
    "  Data is powerful ",
    "Python is GREAT for data analysis ",
    "DATA   analysis is fun"
]

output = [w for s in sentences for w in s.strip().lower().split() if len(w) > 3]
output


['data', 'powerful', 'python', 'great', 'data', 'analysis', 'data', 'analysis']

In [76]:
"""
📊 Practice Problem: Flag Students at Risk
⸻

🎯 Tasks:
	1.	For each student, compute their average score across all subjects.
	2.	Add a new column: "at_risk" — set to True if their average is below 70, else False.
	3.	Return a DataFrame like:

student   avg_score   at_risk
Alice     65.0        True
Bob       71.0        False
Charlie   82.5        False
"""

import pandas as pd

df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob', 'Charlie', 'Alice'],
    'subject': ['Math', 'Math', 'Math', 'English', 'English', 'English', 'History'],
    'score': [85, 72, 90, 60, 70, 75, 50]
})

new_df = df.groupby('student')['score'].mean().reset_index().rename(columns={'score': 'avg_score'})

new_df['at_risk'] = new_df['avg_score'].apply(lambda x: x < 70)
new_df

,student,avg_score,at_risk
0,Alice,65.0,True
1,Bob,71.0,False
2,Charlie,82.5,False


In [ ]:
"""

📊 Practice Problem: Top Scorer per Subject

⸻

🧠 Tasks:
	1.	For each subject, find the student(s) with the highest score.
	2.	Output a DataFrame with columns:

subject | student | score

	3.	Bonus: Sort the result by subject alphabetically.

"""
import pandas as pd

df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob', 'Charlie'],
    'subject': ['Math', 'Math', 'Math', 'English', 'English', 'English'],
    'score': [88, 79, 93, 85, 90, 87]
})
idx = df.groupby('subject')['score'].idxmax()

output = df.loc[idx]
output

,student,subject,score
4,Bob,English,90
2,Charlie,Math,93


In [55]:
"""
🧠 Practice Problem: Match Students with Their Grades

Tasks:
	1.	Merge the two DataFrames so each student is matched with their subjects and scores.
	2.	Filter the result to only include scores ≥ 90.
	3.	Sort the result by score in descending order.

Expected columns:

student_id | name | subject | score
"""
import pandas as pd

students = pd.DataFrame({
    'student_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie']
})

grades = pd.DataFrame({
    'student_id': [1, 1, 2, 3, 3],
    'subject': ['Math', 'English', 'Math', 'English', 'Science'],
    'score': [85, 88, 79, 92, 95]
})

df = pd.merge(students, grades, how='inner', on='student_id')
df[df['score']>=90]

,student_id,name,subject,score
3,3,Charlie,English,92
4,3,Charlie,Science,95


In [52]:
"""
Great — here’s a focused pandas practice that uses filtering, grouping, and basic aggregation.

⸻

📊 Practice Problem: Average Scores by Subject

	1.	Filter the DataFrame to only include scores above 80.
	2.	Group by subject and compute the average score.
	3.	Reset the index of the result so it’s a flat DataFrame.

Expected output structure:

   subject  avg_score
0  English       86.0
1     Math       85.0
2  Science       89.0

"""
import pandas as pd

df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Alice', 'Bob', 'Alice', 'Bob'],
    'subject': ['Math', 'Math', 'Science', 'Science', 'English', 'English'],
    'score': [85, 78, 90, 88, 92, 80]
})

df[df['score'] > 80].groupby('subject', as_index=False)['score'].mean().rename(columns={'score': 'avg_score'})

,subject,avg_score
0,English,92.0
1,Math,85.0
2,Science,89.0


In [42]:
"""
🧠 Practice Problem: Extract Words with More Than 3 Letters

Task:
	1.	Use a list comprehension to extract all words longer than 3 characters.
	2.	Flatten all the words from all sentences into one list.
	3.	Your result should look something like:

['data', 'powerful', 'python', 'great', 'data', 'analysis', 'data', 'analysis']

"""

sentences = [
    "data is powerful",
    "python is great for data analysis",
    "data analysis is fun"
]


output = [x for s in sentences for x in s.split() if len(x)>3]
output

['data', 'powerful', 'python', 'great', 'data', 'analysis', 'data', 'analysis']

In [37]:
"""
	1.	Use a list comprehension to:
	•	Split each sentence into words.
	•	Flatten all the words into a single list.

output
['data', 'is', 'powerful', 'python', 'is', 'great', 'for', 'data', 'analysis', 'data', 'analysis', 'is', 'fun']    
"""

sentences = [
    "data is powerful",
    "python is great for data analysis",
    "data analysis is fun"
]

output = []

# for s in sentences:
#     spl = s.split()
#     for w in spl:
#         output.append(w)

l = [w for s in sentences for w in s.split()]
# l = {w for s in sentences for w in s.split()}

l 

# output

['data',
 'is',
 'powerful',
 'python',
 'is',
 'great',
 'for',
 'data',
 'analysis',
 'data',
 'analysis',
 'is',
 'fun']

In [ ]:
"""
Objective: Write a Python script that counts how many times each word appears in a list of sentences.

output:
{'data': 3, 'is': 3, 'powerful': 1, 'python': 1, 'great': 1, 'for': 1, 'analysis': 2, 'fun': 1}
"""
sentences = [
    "data is powerful",
    "python is great for data analysis",
    "data analysis is fun"
]

output = {}
for s in sentences:
    spl = s.split()
    for w in spl:
        output[w] = output.get(w, 0) + 1
        
# output


"""
Follow up Task:
Using your previous word count output dictionary, do the following with pandas:
	1.	Convert the dictionary into a DataFrame with columns:
	•	"word" (the word)
	•	"count" (how many times it appeared)
	2.	Sort the DataFrame by "count" in descending order.
	3.	Bonus: Filter to show only words that appear more than once.

output:
      word  count
0     data      3
1       is      3
2  analysis      2
...
"""
import pandas as pd

as_df = pd.DataFrame(list(output.items()), columns=['word', 'count']).sort_values(by='count', ascending=False).reset_index(drop=True)
df = as_df[as_df['count'] > 1]
df



,word,count
0,data,3
1,is,3
2,analysis,2


In [10]:
""" 
💻 Problem: Tuple squares

Write a list comprehension that creates a list of 
tuples in the form (n, n²) for numbers from 1 to 10.
...this problem is a variation of the one below

[(1, 1), (2, 4), (3, 9), ..., (10, 100)]
"""
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# result = [(n, n**2) for n in nums]
# result

"""
dictionary version
"""
result = {n:n**2 for n in nums}
result



{1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49, 8: 64, 9: 81, 10: 100}

In [17]:
"""
Write a list comprehension that takes a list of integers 
and returns a new list with the squares of the odd numbers only.

nums = [1, 2, 3, 4, 5, 6, 7]
# result: [1, 9, 25, 49]

"""

nums = [1, 2, 3, 4, 5, 6, 7]

result = [n**2 for n in nums if n % 2 != 0]
result


[1, 9, 25, 49]

In [ ]:
"""
Write a function that takes a sentence and returns a list of all words that:
	1.	Are longer than 3 characters
	2.	Start with a vowel (a, e, i, o, u)
	3.	Are converted to lowercase

sentence = "Apples are excellent, but Oranges often outshine"
result = vowel_words(sentence)
print(result)  # ['apples', 'excellent', 'oranges', 'often', 'outshine']
"""

sentence = "Apples are excellent, but Oranges often outshine Excellent!"

def vowel_words(sentence):
    vowels = ['a','e','i','o','u']
    result = []
    for s in sentence.split():     # i think this should fix it...
        s = s.lower().strip(',.!')
        if (len(s) > 3) and (s[0] in vowels):
            result.append(s)
    return result

vowel_words(sentence)

# Great! Here’s a cleaner version that also:

# ✅ Removes punctuation
# ✅ Handles capitalization
# ✅ Handles edge cases like "Excellent!" or "apples."


['apples', 'excellent', 'oranges', 'often', 'outshine', 'excellent']

In [ ]:
"""
Write a for loop that:
	1.	Converts each temperature to Fahrenheit (F = C * 9/5 + 32)
	2.	Appends only the Fahrenheit values that are above 75°F to a new list called hot_days_f.
"""
# temps = [22, 25, 19, 24, 28, 21, 23]
# hot_days_f = []

# for t in temps:
#     f = (t*(9/5) + 32)
#     if f > 75:
#         hot_days_f.append(f)

# hot_days_f

"""
bonus - append both F and C in a dictionary {C:F}
"""
temps = [22, 25, 19, 24, 28, 21, 23]

hot_days = {}

for t in temps:
    f = (t*(9/5) + 32)
    if f > 75:
        hot_days[t] = f 

hot_days





{25: 77.0, 24: 75.2, 28: 82.4}

In [ ]:
# You’re given a list of filenames. Write a loop that builds a dictionary where:
# 	•	Each key is a file extension (like 'txt', 'csv', 'jpg')
# 	•	Each value is the number of times that extension appears

files = [
    "report.csv", "data.csv", "image.jpg", "notes.txt",
    "summary.txt", "presentation.ppt", "photo.jpg"
]

solution = {}


for f in files:
    extensions = f[-3:]
    solution[extensions] = solution.get(extensions,0) + 1

solution


{'csv': 2, 'jpg': 2, 'txt': 2, 'ppt': 1}

In [ ]:
# Write a loop that:
# 	•	Tallies how many names start with each letter
# 	•	Uses .get() to safely update counts in a dictionary

names = [
    "Alice", "Aaron", "Bob", "Brenda", "Charlie", "Catherine", "David", "Diana"
]

first_letter = []
solution = {}

for n in names:
    k = n[0]
    solution[k] = solution.get(k,0) + 1

solution 

{'A': 2, 'B': 2, 'C': 2, 'D': 2}

In [2]:
# Use a for loop to tally how many votes each candidate received

votes = [
    'Alice', 'Bob', 'Alice', 'Eve', 'Bob', 'Alice', 'Eve', 'Eve', 'Bob', 'Alice'
]

votes_dict = {}

for v in votes:
    votes_dict[v] = votes_dict.get(v, 0) + 1

votes_dict


{'Alice': 4, 'Bob': 3, 'Eve': 3}

In [ ]:
# Task:
# Write a Python function that takes a list of strings (sentences) and returns a dictionary 
# with the frequency of each word (case-insensitive).


sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog barked, and the fox ran away!"
]
s_list = []

for s in sentences:
    s_split = s.split()
    for w in s_split:
        s_list.append(w.lower().strip(',.!'))

s_unique = []
for word in s_list:
    if word not in s_unique:
        s_unique.append(word)

a_dict = {}
for j in s_unique:
    a_dict[j] = s_list.count(j)

a_dict






{'the': 4,
 'quick': 1,
 'brown': 1,
 'fox': 2,
 'jumps': 1,
 'over': 1,
 'lazy': 1,
 'dog': 2,
 'barked': 1,
 'and': 1,
 'ran': 1,
 'away': 1}

In [33]:

vowels = ['a','e','i','o','u']
words = ['apple', 'sky', 'education', 'loop']
solution = []
for w in words:
    counter = 0
    for l in w:
        if l in vowels:
            counter += 1
    solution.append(counter)

solution

[2, 0, 5, 2]

In [39]:
# get longest word in a text
text = "Learning Python with practicing builds strong coding muscle memory"
words = text.split()
lengths = []
for w in words:
    lengths.append(len(w))

biggest = max(lengths)
lengths.index(biggest)


3

In [ ]:
"""
Find the distance between the two furthest apart values in a list.
Create a dist function, receiving a list of numbers (integers or floating points), 
and returning the bigger distance between any two given values, 
so typically the distance between the biggest and the smallest, like:
>>> dist([1, 2, 3])
2
"""

In [ ]:
"""
Write a function named from_roman_numeral that return the value of a given roman numeral.

if next > curr, then -=curr
else +=curr
upper...

"""

roman_str = 'XLIX'

def from_roman_numeral(roman_str):
    conversion = {'I':1, 'V':5, 'X':10, 'L':50 ,'C':100,'M':1000}
    value = 0
    for n in range(len(roman_str)): # determine value, and add to total
        val = conversion[roman_str[n]]
        if n+1 < len(roman_str): 
            next_val = conversion[roman_str[n+1]]
            if val < next_val:
                value -= val
                continue
        value += val

    return(value)

from_roman_numeral(roman_str)
# can this be shortened using tech support?




49

In [ ]:
"""
Write a function named to_roman_numeral that converts integers to roman numerals.

"""

to_numeral = 24

conversion = {1000:'M', 100:'C', 50:'L', 10:'X', 5:'V', 1:'I' }
num_list = list(conversion.keys())
output = ''

while to_numeral > 0:

    max_key = max([x for x in num_list if x <= to_numeral], default=1)
    max_val = conversion[max_key]
    max_key_prior = num_list.index(max_key) - 1      # account for errors when this doesnt exist
    max_val_prior = conversion[num_list[max_key_prior]]

    output += max_val
    if 4 * max_key <= to_numeral:
        output += max_val_prior
        to_numeral -= 4 * max_key
    else:
        to_numeral -= max_key
        
    

output



'XXIV'

In [109]:
"""
You found a safe. There's a numeric keypad on it...
The keypad is dirty, thanks to this you can easily spot that numbers 1, 5, and 8 are frequently used. 
It's so clear that you guess no other digit is in use in this code.
Write a script printing all combinations of 4 digits that use all of those used digits.
Output one combination per line, separate digits using spaces, so your output should look like:
1 1 5 8
1 5 5 8
1 5 8 8
"""

"\nYou found a safe. There's a numeric keypad on it...\nThe keypad is dirty, thanks to this you can easily spot that numbers 1, 5, and 8 are frequently used. \nIt's so clear that you guess no other digit is in use in this code.\nWrite a script printing all combinations of 4 digits that use all of those used digits.\nOutput one combination per line, separate digits using spaces, so your output should look like:\n1 1 5 8\n1 5 5 8\n1 5 8 8\n"

In [ ]:
"""
Write a function combining two strings so they are displayed side by side, separated by a column of pipe (|) symbols, like this:
Lorem ipsum dolor sit amet, consectetur|Lorem ipsum dolor sit amet, consectetur
adipiscing elit. Sed non risus.        |adipiscing elit. Sed non risus.        
Suspendisse lectus tortor, dignissim   |Suspendisse lectus tortor, dignissim   
sit amet, adipiscing nec, utilisez sed |sit amet, adipiscing nec, utilisez sed 
sin dolor.                             |sin dolor. 

This function will also take a 3rd, optional, parameter named width representing the maximum width allowed, defaulting to 79, so the function prototype is:
"""

In [ ]:
"""
Implement the fill_magic_square function.
It takes a single argument: a numpy.array of integers representing a partially filled magic square. Holes are represented using zeros.
Your fill_magic_square function will have to find and fill the gaps in the square.
Do not return a value, just modify the square in-place.
Beware, my magic squares may contain any natural number (> 0), I do not restrict myself to magic squares with numbers from 1 to square_size ** 2.

easy_square = np.array([
    [2, 7, 6],
    [9, 0, 1],
    [4, 3, 8],
])
fill_magic_square(easy_square)
print(easy_square)

Should give:
array([[ 2, 7, 6 ],
       [ 9, 5, 1 ],
       [ 4, 3, 8 ]]])

"""

##### **Hangman ---> Python game by E**

In [ ]:
# num_letters = int(input("welcome to hell...I mean, hangman. Six wrong guesses and I win. hahaha...how many letters do you want the answer to have? "))

# # generate a word with num_letters letters
# answer_string = "doggy"
# answer_list = []
# starting_list = []

# for a in answer_string:
#     answer_list.append(a)

# for s in range(num_letters):
#     starting_list.append("-")

#     # change the above to a string? "-"

# # other variables
# guess_list = []
# chances = 6
# chance_counter = 0


In [ ]:
# guess = input(f"the word is {starting_list}. You get {chances} chances - guess a letter, or guess the word: ")
# if guess == answer_string:
#     print("damn, you're good. I quit!")

# while (guess != answer_string) & (chance_counter < 6) & (starting_list != answer_list):
#     # word is selected
#     if len(guess) > 1:

#     # word is correct
#         if guess == answer_string:
#             print("you WIN!")

#     # word is incorrect
#         else:
#             chance_counter += 1
#             guess = input(f"nope! that's {chance_counter} out of 6 guesses. guess a letter, or the word: {starting_list}")


#     # letter is selected
#     else:
#         if guess in guess_list:
#             guess = input(f"you guessed that letter already - try a different one {starting_list}: ")
#     # letter is correct
#         elif guess in answer_string:
#             yes_list = []
#             guess_list.append(guess)
#             for letter_ind in range(len(answer_list)):
#                 if answer_list[letter_ind] == guess:
#                     yes_list.append(letter_ind)
            
#             for m in yes_list:
#                 starting_list[m] = guess
#             if starting_list == answer_list:
#                 print("well played. I concede - you win.")        
#                 break  
#             else:
#                 guess = input(f"Sure, that letter's in there: {starting_list}. guess a letter, or the word: ")

    
#     # letter is incorrect
#         else:
#             guess_list.append(guess)
            
#             if chance_counter == 6:
#                 print("Got 'eeeeem! I win")
#                 break
#             else:
#                 guess = input(f"nope! that's {chance_counter} out of 6 guesses. guess a letter, or the word: {starting_list}")
#                 chance_counter += 1




well played. I concede - you win.


In [ ]:
# # Provide a script that prints the number of words in the given paragraph.
# # I prefilled the answer box with the paragraph, in a variable, but in case you lose it, here it is:

# #   - remove punctuation
# #   - all lowercase

# # whetting_your_appetite = "Python is an easy to learn, powerful programming language. It has efficient high level data structures and a simple but effective approach to object oriented programming. This tutorial introduces the reader informally to the basic concepts and features of the Python language and system. For a description of standard objects and modules..."
# word_count = "I am using Python. This is a string with has words. I am counting the words, words, words, using python python python python python"

# word_split = word_count.split()
# counter = 0

# # count total words
# word_list = []
# for word in word_split:
#     word_list.append(word.lower().strip(".,"))
#     counter += 1

# counter
# # word_list

# # can you also count the number of each word - output as a list

# deduped = []
# for i in word_list:
#     if i not in deduped:
#         deduped.append(i)
#     else:
#         deduped = deduped

# # deduped

# word_dict = {}

# for k in deduped:
#     val = 0
#     for v in word_list:
#         if k == v:
#             val += 1
#         word_dict[k] = val

# word_dict





24

In [ ]:
# # Ask the user for a number and determine whether the number is prime or not. 

# num = int(input("let's check a numbers prime-ability. enter one: "))
# counter = 0
# factors = []
# for i in range(1, num + 1):
#     if num % i == 0:
#         counter += 1
#         factors.append(i)
#     else:
#         counter = counter

# print(counter)
# print(factors)
# if counter <= 2:
#     print("PRIME")
# else:
#     print("unprime")


4
[1, 3, 127, 381]
unprime
